<a href="https://colab.research.google.com/github/ramdanibili46/EcoSortAI/blob/main/waste_classification_9class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Waste Classification Model — 9 Kelas

**Dataset lokal** dengan struktur direktori:
```
dataset/
├── Kaca/
├── Kardus/
├── Kertas/
├── Logam/
├── Plastik/
├── Residu/
├── hazardous/
├── organic/
└── recyclable/
```

> **Tidak perlu download dari link** — model ini langsung membaca folder `dataset/` di direktori lokal.

## 1. Dependencies

In [ ]:
import os
import glob
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').disabled = True

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU'))} GPU(s)")

TensorFlow version: 2.20.0
GPU available: 0 GPU(s)


## 2. Konfigurasi Path & Parameter

In [ ]:
# ============================================================
# UBAH PATH INI sesuai lokasi folder dataset kamu
# ============================================================
BASE_DIR    = "dataset"          # folder utama yang berisi 9 subfolder kelas
MODEL_PATH  = "best_model.h5"   # path untuk menyimpan model terbaik
IMG_SIZE    = (224, 224)         # ukuran input EfficientNetB0
BATCH_SIZE  = 32
EPOCHS      = 30
SEED        = 42

# Kelas sesuai nama subfolder
CLASS_NAMES = sorted(os.listdir(BASE_DIR))
NUM_CLASSES = len(CLASS_NAMES)

print(f"Ditemukan {NUM_CLASSES} kelas:")
for i, c in enumerate(CLASS_NAMES):
    print(f"  [{i}] {c}")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset'

## 3. Eksplorasi Data

In [ ]:
# Hitung jumlah gambar per kelas
class_counts = {}
for cls in CLASS_NAMES:
    images = glob.glob(os.path.join(BASE_DIR, cls, '*.jpg')) + \
             glob.glob(os.path.join(BASE_DIR, cls, '*.jpeg')) + \
             glob.glob(os.path.join(BASE_DIR, cls, '*.png'))
    class_counts[cls] = len(images)

total = sum(class_counts.values())
print(f"Total gambar: {total}\n")
for cls, count in class_counts.items():
    print(f"  {cls:15s}: {count:5d} gambar ({count/total*100:.1f}%)")

In [ ]:
# Visualisasi distribusi kelas
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(class_counts.keys(), class_counts.values(),
              color=plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES)))
ax.set_title('Distribusi Jumlah Gambar per Kelas', fontsize=14, fontweight='bold')
ax.set_xlabel('Kelas', fontsize=12)
ax.set_ylabel('Jumlah Gambar', fontsize=12)
ax.bar_label(bars, padding=3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Tampilkan contoh gambar dari setiap kelas
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.flatten()

for idx, cls in enumerate(CLASS_NAMES):
    img_files = glob.glob(os.path.join(BASE_DIR, cls, '*.jpg')) + \
                glob.glob(os.path.join(BASE_DIR, cls, '*.jpeg')) + \
                glob.glob(os.path.join(BASE_DIR, cls, '*.png'))
    if img_files:
        sample_img = load_img(img_files[0], target_size=IMG_SIZE)
        axes[idx].imshow(sample_img)
        axes[idx].set_title(cls, fontsize=12, fontweight='bold')
        axes[idx].axis('off')

plt.suptitle('Contoh Gambar per Kelas', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Data Augmentation & Split

Karena dataset hanya memiliki **satu folder** (tanpa Train/Test), kita:
- Pakai `validation_split=0.2` di `ImageDataGenerator` untuk memisahkan train & validasi
- Pisahkan 10% secara manual untuk test set menggunakan `flow_from_dataframe`

In [ ]:
# Kumpulkan semua path gambar dan labelnya
all_images = []
all_labels = []

for cls in CLASS_NAMES:
    img_files = glob.glob(os.path.join(BASE_DIR, cls, '*.jpg')) + \
                glob.glob(os.path.join(BASE_DIR, cls, '*.jpeg')) + \
                glob.glob(os.path.join(BASE_DIR, cls, '*.png'))
    for f in img_files:
        all_images.append(f)
        all_labels.append(cls)

df = pd.DataFrame({'filepath': all_images, 'label': all_labels})
print(f"Total dataset: {len(df)} gambar")

# Split: 80% train+val, 20% test
train_val_df, test_df = train_test_split(
    df, test_size=0.10, stratify=df['label'], random_state=SEED
)
print(f"Train+Val : {len(train_val_df)} gambar")
print(f"Test      : {len(test_df)} gambar")

In [ ]:
# Data generators
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    zoom_range=0.3,
    rotation_range=15,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

# Train generator
train_ds = train_datagen.flow_from_dataframe(
    dataframe=train_val_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='training',
    shuffle=True,
    seed=SEED
)

# Validation generator
valid_ds = train_datagen.flow_from_dataframe(
    dataframe=train_val_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='validation',
    shuffle=False,
    seed=SEED
)

# Test generator
test_ds = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\nClass indices:", train_ds.class_indices)

In [ ]:
# Visualisasi hasil augmentasi
sample_batch = next(train_ds)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()
label_map = {v: k for k, v in train_ds.class_indices.items()}

for i in range(10):
    axes[i].imshow(sample_batch[0][i])
    label_idx = np.argmax(sample_batch[1][i])
    axes[i].set_title(label_map[label_idx], fontsize=10)
    axes[i].axis('off')

plt.suptitle('Contoh Hasil Augmentasi Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Membangun Model

Menggunakan **EfficientNetB0** sebagai base model (transfer learning) — lebih ringan dan akurat dibanding VGG16 untuk multi-class classification.

In [ ]:
# Callbacks
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    MODEL_PATH,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

callback_list = [early_stopping, checkpoint, reduce_lr]

In [ ]:
# Base model EfficientNetB0
base_model = EfficientNetB0(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze semua layer base model dulu (Phase 1: feature extraction)
base_model.trainable = False

print(f"Total layers di base model: {len(base_model.layers)}")
print(f"Trainable parameters: {base_model.count_params():,}")

In [ ]:
# Build model lengkap
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(512, activation='relu', kernel_initializer='he_uniform'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(256, activation='relu', kernel_initializer='he_uniform'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')  # 9 kelas
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()

## 6. Training Model — Phase 1 (Feature Extraction)

In [ ]:
history_phase1 = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=valid_ds,
    callbacks=callback_list,
    verbose=1
)

## 7. Fine-Tuning — Phase 2

Unfreeze sebagian layer base model untuk fine-tuning dengan learning rate lebih kecil.

In [ ]:
# Unfreeze 30 layer terakhir dari base model
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Layer yang di-unfreeze: {trainable_count} dari {len(base_model.layers)}")

# Recompile dengan learning rate lebih kecil
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_phase2 = model.fit(
    train_ds,
    epochs=15,
    validation_data=valid_ds,
    callbacks=callback_list,
    verbose=1
)

## 8. Visualisasi Training History

In [ ]:
# Gabungkan history Phase 1 & Phase 2
def merge_history(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

full_history = merge_history(history_phase1, history_phase2)
history_df = pd.DataFrame(full_history)
history_df.to_csv('training_history.csv', index=False)
print("History disimpan ke training_history.csv")
history_df.tail()

In [ ]:
# Plot Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(full_history['loss'], color='deeppink', linewidth=2.5, label='Train')
axes[0].plot(full_history['val_loss'], color='dodgerblue', linewidth=2.5, label='Validation')
axes[0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot Accuracy
axes[1].plot(full_history['accuracy'], color='deeppink', linewidth=2.5, label='Train')
axes[1].plot(full_history['val_accuracy'], color='dodgerblue', linewidth=2.5, label='Validation')
axes[1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Evaluasi Model pada Test Set

In [ ]:
# Load model terbaik
best_model = load_model(MODEL_PATH)

# Evaluasi
test_loss, test_acc, test_auc = best_model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Test AUC      : {test_auc:.4f}")

In [ ]:
# Prediksi seluruh test set
test_ds.reset()
y_pred_proba = best_model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_ds.classes

label_map = {v: k for k, v in train_ds.class_indices.items()}
target_names = [label_map[i] for i in range(NUM_CLASSES)]

print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=target_names))

In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].tick_params(axis='x', rotation=30)

# Percentage
sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Actual', fontsize=12)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Classification Report Heatmap
from sklearn.metrics import classification_report
import json

report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
report_df = pd.DataFrame(report).T.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
report_df = report_df[['precision', 'recall', 'f1-score']]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(report_df.astype(float), annot=True, fmt='.3f', cmap='YlOrRd',
            vmin=0, vmax=1, ax=ax)
ax.set_title('Classification Report per Kelas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Prediksi Gambar Baru

In [ ]:
def predict_image(img_path, model, class_names, img_size=(224, 224)):
    """
    Prediksi kategori sampah dari satu gambar.

    Args:
        img_path: path ke file gambar
        model: model yang sudah ditraining
        class_names: list nama kelas
        img_size: ukuran input model
    """
    img = load_img(img_path, target_size=img_size)
    img_array = img_to_array(img) / 255.0
    img_batch = np.expand_dims(img_array, axis=0)

    proba = model.predict(img_batch, verbose=0)[0]
    pred_idx = np.argmax(proba)
    pred_class = class_names[pred_idx]
    confidence = proba[pred_idx] * 100

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(img)
    axes[0].set_title(f'Prediksi: {pred_class}\nKepercayaan: {confidence:.1f}%',
                      fontsize=13, fontweight='bold',
                      color='green' if confidence > 70 else 'orange')
    axes[0].axis('off')

    colors = ['#2196F3' if i == pred_idx else '#E0E0E0' for i in range(len(class_names))]
    axes[1].barh(class_names, proba * 100, color=colors)
    axes[1].set_xlabel('Probabilitas (%)')
    axes[1].set_title('Distribusi Probabilitas', fontsize=12)
    axes[1].set_xlim(0, 100)

    for i, v in enumerate(proba * 100):
        axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    return pred_class, confidence

In [ ]:
# Contoh: ambil satu gambar dari test set untuk dicoba
sample_test_images = test_df.sample(6, random_state=SEED)['filepath'].values

for img_path in sample_test_images:
    actual_class = os.path.basename(os.path.dirname(img_path))
    print(f"File    : {img_path}")
    print(f"Aktual  : {actual_class}")
    pred, conf = predict_image(img_path, best_model, target_names)
    status = '✅ BENAR' if pred == actual_class else '❌ SALAH'
    print(f"Status  : {status}\n")

## 11. Simpan Model Final

In [ ]:
# Simpan dalam format SavedModel (lebih portable)
best_model.save('waste_classifier_9class')
print("Model disimpan ke folder: waste_classifier_9class/")

# Simpan juga class names
import json
with open('class_names.json', 'w') as f:
    json.dump(target_names, f, indent=2)
print("Class names disimpan ke: class_names.json")

print("\n=== Summary Akhir ===")
print(f"Jumlah kelas    : {NUM_CLASSES}")
print(f"Kelas           : {target_names}")
print(f"Test Accuracy   : {test_acc*100:.2f}%")
print(f"Test AUC        : {test_auc:.4f}")